In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import torch
from scripts.utils.load_model import load_model
from scripts.utils.load_model_torchvision import load_torchvision_model 
from scripts.utils.load_dataset import load_dataset
from ulib.data import TensorLoader

# model, dl_train, dl_eval = load_robust_experiment("Standard", "cifar10")


model, trans = load_torchvision_model("vgg16", device="cuda")
dl_train, dl_eval, dl_test = load_dataset("imagenet", trans, train_batch=128, eval_batch=128)

# take a subset of dl_train of size 500

orig_tensors: tuple[torch.Tensor, ...] = dl_train.tensors
# sample a subset of size 500, based on the same indices for all tensors
indices = torch.randperm(orig_tensors[0].shape[0])[:500]
subset_tensors = tuple(tensor[indices] for tensor in orig_tensors)

dl_train = TensorLoader(subset_tensors, batch_size=dl_train.batch_size, shuffle=dl_train.shuffle, drop_last=False, transforms=dl_train.transforms)

In [ ]:
from ulib import StopCriteria, PertModule
from ulib.attacks.dm_uap import DM_UAP

eps = 10 / 255
stop = StopCriteria(max_epochs=20, max_time=60 * 30)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=eps, random_init=True)

attack = DM_UAP(
    pert_model=pert_model,
    total_epochs=20,
    beta=9,
    theta_neighborhood=1.0,
    data_neighborhood=32.0,
    data_steps=10,
    theta_steps=10,
    smooth_rate=0.2,
    eval_freq=1,
)

pert = attack.fit(dl_train, dl_eval, stop)

In [ ]:
from ulib.evaluator import ExtendedEvaluator
from ulib.utils.visualize import display_pert

evaluator = ExtendedEvaluator(pert_model, verbose=True)
print(evaluator.evaluate(dl_eval))
display_pert(pert_model)